# Task 2 Create a SHAP Explainer

Train and evaluate a Random Forest, then create and verify stroke-class SHAP explanations. Run all cells from top to bottom using the environment installed from `requirements.txt`. See `README.md` for setup instructions.


## 1 Imports and dataset preview

Load the original stroke dataset used in Task 1.


In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import shap
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import RandomOverSampler

SEED = 2021
candidates = [Path.cwd(), Path.cwd().parent]
PROJECT = next(
    (folder for folder in candidates
     if (folder / "data/healthcare-dataset-stroke-data.csv").exists()),
    None,
)
if PROJECT is None:
    raise FileNotFoundError("Download the dataset into data/ and run from the submission folder.")

data = pd.read_csv(PROJECT / "data/healthcare-dataset-stroke-data.csv")
print("Python:", sys.version.split()[0])
print("SHAP:", shap.__version__)
print("Dataset shape:", data.shape)
display(data.head())


/Users/benjy/UM/XAI/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: 3.9.6
SHAP: 0.49.1
Dataset shape: (5110, 12)


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


## 2 Training and test split

Remove the identifier and separate the target. Use a reproducible, stratified 80:20 split.


In [2]:
X = data.drop(columns=["id", "stroke"])
y = data["stroke"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED,
)
X_train = X_train.copy()
X_test = X_test.copy()

split_counts = pd.DataFrame({
    "Training": y_train.value_counts(),
    "Test": y_test.value_counts(),
}).reindex([0, 1])
split_counts.index.name = "stroke"
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
display(split_counts)


Training rows: 4088
Test rows: 1022


,Training,Test
stroke,,
0,3889,972
1,199,50


## 3 Preprocess the features

Fill missing BMI using the training median, one-hot encode categories, and align test columns with training columns.


In [3]:
bmi_median = X_train["bmi"].median()
X_train["bmi"] = X_train["bmi"].fillna(bmi_median)
X_test["bmi"] = X_test["bmi"].fillna(bmi_median)

X_train = pd.get_dummies(X_train, dtype=float)
X_test = pd.get_dummies(X_test, dtype=float)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

print("Training BMI median:", bmi_median)
print("Encoded training shape:", X_train.shape)
print("Encoded test shape:", X_test.shape)
print("Missing values in training:", X_train.isna().sum().sum())
print("Missing values in test:", X_test.isna().sum().sum())


Training BMI median: 28.1
Encoded training shape: (4088, 21)
Encoded test shape: (1022, 21)
Missing values in training: 0
Missing values in test: 0


## 4 Oversample the training set

Balance the training classes by duplicating stroke records. Keep the test set unchanged.


In [4]:
sampler = RandomOverSampler(random_state=SEED)
X_train_balanced, y_train_balanced = sampler.fit_resample(X_train, y_train)

balance_counts = pd.DataFrame({
    "Before oversampling": y_train.value_counts(),
    "After oversampling": y_train_balanced.value_counts(),
}).reindex([0, 1])
balance_counts.index.name = "stroke"
display(balance_counts)
print("Balanced training shape:", X_train_balanced.shape)
print("Unchanged test shape:", X_test.shape)


/Users/benjy/UM/XAI/.venv/lib/python3.9/site-packages/sklearn/base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
/Users/benjy/UM/XAI/.venv/lib/python3.9/site-packages/sklearn/base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(


,Before oversampling,After oversampling
stroke,,
0,3889,3889
1,199,3889


Balanced training shape: (7778, 21)
Unchanged test shape: (1022, 21)


## 5 Train the Random Forest

Fit 100 trees using the balanced training data and a fixed random seed.


In [5]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=SEED,
    n_jobs=-1,
)
rf.fit(X_train_balanced, y_train_balanced)
y_pred = rf.predict(X_test)
print("Random Forest training finished.")


Random Forest training finished.


## 6 Evaluate the model

Report accuracy, macro F1, class-level metrics and a confusion matrix. Compare with always predicting no stroke.


In [6]:
majority_pred = np.zeros(len(y_test), dtype=int)
evaluation = pd.DataFrame({
    "Accuracy": [
        accuracy_score(y_test, y_pred),
        accuracy_score(y_test, majority_pred),
    ],
    "Macro F1": [
        f1_score(y_test, y_pred, average="macro"),
        f1_score(y_test, majority_pred, average="macro", zero_division=0),
    ],
}, index=["Random Forest", "Always predict no stroke"])
display(evaluation.round(4))

print("Classification report:")
print(classification_report(
    y_test, y_pred, labels=[0, 1],
    target_names=["No stroke", "Stroke"], zero_division=0,
))
confusion = pd.DataFrame(
    confusion_matrix(y_test, y_pred, labels=[0, 1]),
    index=["Actual no stroke", "Actual stroke"],
    columns=["Predicted no stroke", "Predicted stroke"],
)
display(confusion)


,Accuracy,Macro F1
Random Forest,0.9423,0.5312
Always predict no stroke,0.9511,0.4875


Classification report:
              precision    recall  f1-score   support

   No stroke       0.95      0.99      0.97       972
      Stroke       0.20      0.06      0.09        50

    accuracy                           0.94      1022
   macro avg       0.58      0.52      0.53      1022
weighted avg       0.92      0.94      0.93      1022



,Predicted no stroke,Predicted stroke
Actual no stroke,960,12
Actual stroke,47,3


Accuracy is **94.23%**, but only **3 of 50 stroke cases** are detected (6% recall). High overall accuracy therefore hides poor stroke detection.


## 7 Create the SHAP explainer

Tree SHAP suits the Random Forest's tree structure. Explain all test patients and select the stroke class (`1`). Positive contributions raise the predicted stroke probability relative to the baseline; negative contributions lower it.


In [7]:
explainer = shap.TreeExplainer(rf)
shap_values = explainer(X_test)

stroke_class_index = list(rf.classes_).index(1)
stroke_shap = shap_values[:, :, stroke_class_index]

print("Model classes:", rf.classes_)
print("All-class SHAP shape:", shap_values.values.shape)
print("Test data shape:", X_test.shape)
print("Stroke SHAP shape:", stroke_shap.values.shape)


Model classes: [0 1]
All-class SHAP shape: (1022, 21, 2)
Test data shape: (1022, 21)
Stroke SHAP shape: (1022, 21)


## 8 Verify the SHAP explanations

Check that **baseline + sum of contributions = predicted stroke probability** for every test patient, then inspect one patient.


In [8]:
stroke_probabilities = rf.predict_proba(X_test)[:, stroke_class_index]
reconstructed = stroke_shap.base_values + stroke_shap.values.sum(axis=1)
shap_matches = np.allclose(reconstructed, stroke_probabilities, atol=1e-6)

print("SHAP matches model probabilities:", shap_matches)
print("Maximum absolute reconstruction error:",
    np.max(np.abs(reconstructed - stroke_probabilities)))

patient_index = 0
patient_summary = pd.Series({
    "Original dataset index": X_test.index[patient_index],
    "Actual label": y_test.iloc[patient_index],
    "Predicted label": y_pred[patient_index],
    "Stroke probability": stroke_probabilities[patient_index],
    "SHAP baseline": stroke_shap.base_values[patient_index],
    "Sum of contributions": stroke_shap.values[patient_index].sum(),
    "Reconstructed probability": reconstructed[patient_index],
}, name="First test patient")
display(patient_summary.to_frame())


SHAP matches model probabilities: True
Maximum absolute reconstruction error: 3.219646771412954e-15


,First test patient
Original dataset index,4926.000000
Actual label,0.000000
Predicted label,0.000000
Stroke probability,0.010000
SHAP baseline,0.500476
Sum of contributions,-0.490476
Reconstructed probability,0.010000


## Task 2 complete

The model is trained and evaluated, Tree SHAP explanations are created, and the reconstruction check passes. Retain `stroke_shap` for the force plot in Task 3 and summary plot in Task 4. Detailed methods and interpretation belong in the report.
